# Fitting Jlens on two types of prompts

I compare two Jlens fitting methods in this research: fitting on raw LLMail-Inject prompts, and fitting on synthetic office prompts with an explicit classification instruction.

The Hugging Face model provides the transformer, the Jlens model wrapper gives access to its layers, and the fitted Jacobian lens contains the matrices used to estimate the final hidden state vectors. This notebook explains the fitting process and displays the settings and commands for both methods.

- [The fitting settings](#The-fitting-settings)
- [How is the Jacobian matrix computed?](#How-is-the-Jacobian-matrix-computed?)
- [The commands for both fitting methods](#The-commands-for-both-fitting-methods)
- [How is the fitted lens used for selection?](#How-is-the-fitted-lens-used-for-selection?)


## The fitting settings

Both fits use 100 prompts with a maximum fitting length of 128 tokens, while the language model's weights stay fixed. The dictionary variable `records` contains each fit's original settings, including the prompt format, inspected layers and hidden state vector size.


In [ ]:
# compare the prompts and parameters used to fit both Jacobian lenses
from pathlib import Path
import ast
import json
import shlex

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'scripts/reproduction/lens_settings.json').exists())
records = json.loads((root / 'scripts/reproduction/lens_settings.json').read_text())

for method, record in records.items():
    print(method)
    print(json.dumps(record['config'], indent=2))
    print('Inspected layers:', record['source_layers'])
    print('Hidden state vector size:', record['d_model'])


## How is the Jacobian matrix computed?

For each fitting prompt, the model first runs a forward pass, then automatic differentiation measures how a small change in an earlier hidden state vector value affects values at the final layer.

Collecting these rates of change forms a Jacobian matrix, and combining the derivatives across selected token positions and averaging across fitting prompts gives one matrix for each inspected layer. The function `fit()` below passes the model and prepared prompts to the pinned Jlens implementation, which defines the token positions used for fitting.


In [ ]:
# display how the fitter passes the model and fitting prompts to Jlens
source = (root / 'scripts/Jacobian_Lens_Fitting.py').read_text()
function = next(node for node in ast.parse(source).body if isinstance(node, ast.FunctionDef) and node.name == 'fit')
print(ast.get_source_segment(source, function))


## The commands for both fitting methods

The list variable `command` assembles the arguments needed for each fit, using the original prompt format and fitting parameters.

The raw template passes LLMail-Inject text directly, while the preformatted template keeps the classification instruction already included in the synthetic prompts. The code prints both commands for inspection, with the exact settings requiring separate approval before a new fit starts.


In [ ]:
for method, record in records.items():
    config = record['config']
    command = ['python', '-m', 'scripts.Jacobian_Lens_Fitting']
    names = {'model_path': '--model-path', 'data_path': '--data', 'output_path': '--output', 'checkpoint_path': '--checkpoint', 'text_column': '--text-column', 'split_column': '--split-column', 'split_value': '--split-value', 'num_prompts': '--num-prompts', 'seed': '--seed', 'template': '--template', 'max_seq_len': '--max-seq-len', 'dim_batch': '--dim-batch', 'checkpoint_every': '--checkpoint-every', 'device': '--device', 'dtype': '--dtype'}
    for name, flag in names.items():
        if config[name] is not None:
            command.extend([flag, str(config[name])])
    # print each complete command so its settings can be reviewed before fitting
    print(method)
    print(shlex.join(command))


## How is the fitted lens used for selection?

After fitting the lens, I use FIPI validation prompts to choose which layer produces the highest average injection and safe accuracy. The function `fit_lens()` performs this layer comparison using an existing fitted lens.

At each layer, multiplying the hidden state vector by its Jacobian matrix estimates the final hidden state vector, then normalization and the unembedding matrix produce the label logits. Converting those logits into injection and safe probabilities allows the same pairwise selection score to be used for selecting FIPI data.


In [ ]:
# inspect layer selection using an existing fitted Jacobian lens
source = (root / 'scripts/curation/selection/data_curation_selection.py').read_text()
function = next(node for node in ast.parse(source).body if isinstance(node, ast.FunctionDef) and node.name == 'fit_lens')
print(ast.get_source_segment(source, function))
